# PyExoCross ExoMol Result API examples

This notebook demonstrates `px.load`, conversion, and every calculation API using the MgH parameters from `test_api_exomol.py`. It is intended for interactive use in Jupyter or Spyder.

Most calculations use `output="memory"` and `log="none"`. Spectral examples use `0–100 cm⁻¹` so they run quickly. Conversion is the exception: it is a format transformation and supports file output only.

## 1. Setup and shared parameters

Update `read_path` and `save_path` if the database is stored elsewhere.

In [1]:
from pathlib import Path
import sys
import numpy as np

start = Path.cwd().resolve()
project_root = next(path for path in (start, *start.parents) if (path / 'src' / 'pyexocross').is_dir())
sys.path.insert(0, str(project_root / 'src'))

import pyexocross as px
print('PyExoCross version:', px.__version__)

PyExoCross version: 1.1.14


In [2]:
COMMON = dict(
    database='ExoMol',
    molecule='MgH',
    isotopologue='24Mg-1H',
    dataset='XAB',
    species_id=501,
    read_path='/Users/beryl/Academic/UCL/PhD/Data/database/ExoMol/',
    save_path='/Users/beryl/Academic/UCL/PhD/Data/pyexocross/',
    logs_path='/Users/beryl/Academic/UCL/PhD/Data/pyexocross/log/MgH_ExoMol_parquet.log',
    cache='parquet',
    verbose=True,
)

COMPUTE_PARAMS = dict(
    ncputrans=4,
    ncpufiles=1,
    chunk_size=500000,
    device='CPU',
)

RANGE_PARAMS = dict(
    wn_wl='WN',
    wn_wl_unit='cm-1',
    min_range=0,
    max_range=30000,
    unc_filter=0.5,
)

MEMORY = dict(output='memory', log='none', verbose=False)
FAST_RANGE = dict(min_range=0, max_range=100)

## 2. Load reusable data

`px.load()` returns `LoadedData`, not `Result`. The same loaded object can be passed to all calculations. Calculations that require every transition file expand the loaded data automatically.

In [3]:
data = px.load(
    **COMMON,
    **COMPUTE_PARAMS,
    **RANGE_PARAMS,
    log='none',
)

print(type(data))
print('database:', data.config.database)
print('loaded range:', data.config.min_wn, data.config.max_wn, 'cm-1')

=== Host CPU Device Info ===
Python: 3.10.8
Platform: macOS-26.5-arm64-arm-64bit
Machine: arm64
Processor: arm
CPU model: arm
Logical cores: 10

╒════════════════╤════════════════════════╕
│ Field          │ Value                  │
╞════════════════╪════════════════════════╡
│ Requested mode │ CPU                    │
├────────────────┼────────────────────────┤
│ Active mode    │ CPU                    │
├────────────────┼────────────────────────┤
│ Provider       │                        │
├────────────────┼────────────────────────┤
│ Reason         │ GPU mode not requested │
╘════════════════╧════════════════════════╛

╒════════════╤════════════╤════════════════╤═══════════╕
│ Database   │ Molecule   │ Isotopologue   │ Dataset   │
╞════════════╪════════════╪════════════════╪═══════════╡
│ ExoMol     │ MgH        │ 24Mg-1H        │ XAB       │
╘════════════╧════════════╧════════════════╧═══════════╛

Loading reusable line-list data ...
Number of all transitions files               : 

## 3. Conversion

Conversion writes a HITRAN-format file and returns `None`. Run this cell only when a converted file is wanted.

In [5]:
# Uncomment to write the converted file. Conversion has no memory Result.
converted = px.conversion(
    data=data,
    conversion_format='HITRAN',
    conversion_min_freq=0,
    conversion_max_freq=30000,
    conversion_unc=None,
    conversion_threshold=None,
    global_qn_label_list=['ElecState', 'v', 'Omega'],
    global_qn_format_list=['%9s', '%2d', '%4s'],
    local_qn_label_list=['J', 'e/f'],
    local_qn_format_list=['%5.1f', '%2s'],
    output='files',
    log='none',
)
# assert converted is None

Reading states from Parquet cache: /Users/beryl/Academic/UCL/PhD/Data/database/ExoMol/MgH/24Mg-1H/XAB/.pyexocross_cache/24Mg-1H__XAB.states.parquet
Number of all transitions files               : 1
Number of selected transitions files          : 1
Number of all decompressed transitions files  : 0
Number of new decompressed transitions files  : 0
Convert data format from ExoMol to HITRAN.
Wavenumber range selected : 0 cm⁻¹ - 30000 cm⁻¹
Selected global quantum number labels : ElecState  v    Omega  
Selected global quantum number formats: %9s        %2d  %4s    
Selected  local quantum number labels : J      e/f  
Selected  local quantum number formats: %5.1f  %2s  

Reading transitions and converting data format from ExoMol to HITRAN ...
[          ]   0%
[█         ]  10%
[██        ]  20%
[███       ]  30%
[████      ]  40%
[█████     ]  50%
[██████    ]  60%
[███████   ]  70%
[████████  ]  80%
[█████████ ]  90%
[██████████] 100%
Processeing transitions file: 24Mg-1H__XAB__00000-30000

## 4. Partition functions

Scalar-table calculations have one unconditional result, so use `result.select()` without T or P.

In [6]:
partition_result = px.partition_functions(
    data=data, ntemp=100, tmax=5000, **MEMORY
)
partition_table = partition_result.select()
partition_result.summary()
print(partition_table.head())
print('temperature grid:', partition_result.coords['temperature'][:5])
print('units:', partition_result.units)
print('parameters:', partition_result.params.to_dict())

partition_functionResult
  coordinates: temperature (50 points)
  conditions: [Condition(T=None, P=None, Tvib=None, Trot=None)]
Calculation parameters
  cache             : parquet
  database          : ExoMol
  dataset           : XAB
  isotopologue      : 24Mg-1H
  max_memory        : 512
  molecule          : MgH
  ntemp             : 100
  output            : memory
  read_path         : /Users/beryl/Academic/UCL/PhD/Data/database/ExoMol/
  refresh_cache     : False
  tmax              : 5000
Timing
  calculate         : 0.002439 s
  load              : 0.000000 s
  prepare           : 0.000000 s
  save              : 0.000000 s
  total             : 0.003441 s
     T  Partition function
0  100  49.892457411409836
1  200   98.56777122985167
2  300  147.55017802996713
3  400   197.5539534393839
4  500  249.70143005442924
temperature grid: [100 200 300 400 500]
units: {'temperature': 'K', 'data': 'dimensionless'}
parameters: {'refresh_cache': False, 'cache': 'parquet', 'database': 'E

## 5. Specific heats

In [7]:
specific_result = px.specific_heats(
    data=data, ntemp=100, tmax=5000, **MEMORY
)
specific_table = specific_result.select()
print(specific_result)
print(specific_table.head())

specific_heatResult
  coordinates: temperature (50 points)
  conditions: [Condition(T=None, P=None, Tvib=None, Trot=None)]
Calculation parameters
  cache             : parquet
  database          : ExoMol
  dataset           : XAB
  isotopologue      : 24Mg-1H
  max_memory        : 512
  molecule          : MgH
  ntemp             : 100
  output            : memory
  read_path         : /Users/beryl/Academic/UCL/PhD/Data/database/ExoMol/
  refresh_cache     : False
  tmax              : 5000
Timing
  calculate         : 0.008462 s
  load              : 0.000000 s
  prepare           : 0.000000 s
  save              : 0.000000 s
  total             : 0.009528 s
     T       Specific heat
0  100   29.12702156977695
1  200  29.181994799107013
2  300  29.600042503720687
3  400   30.54642011089903
4  500    31.7063933331858


## 6. Cooling functions

Cooling functions require all transitions. The first call may expand the reusable data.

In [8]:
cooling_result = px.cooling_functions(
    data=data, ntemp=100, tmax=5000, **MEMORY
)
cooling_table = cooling_result.select()
print(cooling_table.head())
print('timing:', cooling_result.timing)

     T  cooling function
0  100      1.522841e-17
1  200      1.197437e-16
2  300      1.696262e-15
3  400      8.838941e-15
4  500      2.460490e-14
timing: {'calculate': {'cpu': 0.07121699999999986, 'system': 0.040506839752197266}, 'load': {'cpu': 0.0, 'system': 0.0}, 'prepare': {'cpu': 0.0, 'system': 0.0}, 'save': {'cpu': 0.0, 'system': 0.0}, 'total': {'cpu': 0.07159000000000049, 'system': 0.04086494445800781}}


## 7. Lifetimes

In [9]:
lifetime_result = px.lifetimes(
    data=data, compress=False, **MEMORY
)
lifetime_table = lifetime_result.select()
print(lifetime_table.head())
print('coordinates:', lifetime_result.coords.keys())
print('units:', lifetime_result.units)

   ID            E  gtot    J       unc           tau  gfactor +/- e/f  \
0   1     0.000000     4  0.5  0.000001           Inf  2.00231   +   e   
1   2  1431.977929     4  0.5  0.001010    1.5654E-02  2.00231   +   e   
2   3  2800.678343     4  0.5  0.002372    8.8248E-03  2.00231   +   e   
3   4  4102.331435     4  0.5  0.005010    6.6588E-03  2.00231   +   e   
4   5  5331.390135     4  0.5  0.011010    5.6330E-03  2.00231   +   e   

    ElecState  v  hunda:Lambda  hunda:Sigma  hunda:Omega Auxiliary:SourceType  \
0  X(2SIGMA+)  0             0            0            0                   Ma   
1  X(2SIGMA+)  1             0            0            0                   Ma   
2  X(2SIGMA+)  2             0            0            0                   Ma   
3  X(2SIGMA+)  3             0            0            0                   Ma   
4  X(2SIGMA+)  4             0            0            0                   Ma   

   Auxiliary:Ecal  
0        0.000000  
1     1432.066979  
2     28

## 8. Oscillator strengths

In [10]:
oscillator_result = px.oscillator_strengths(
    data=data,
    gf_or_f='f',
    **FAST_RANGE,
    plot=False,
    **MEMORY,
)
oscillator_table = oscillator_result.select()
print(oscillator_table.head())
print(oscillator_result.params)

    uid   lid            os         v
0  4664  4550  4.490456e-33  1.242966
1  4664  4320  4.747903e-30  1.252055
2  4894  4550  5.105924e-30  1.337756
3  8343  8229  5.969027e-34  1.569618
4  8343  7999  3.133132e-30  1.597018
Calculation parameters
  cache             : parquet
  chunk_size        : 500000
  database          : ExoMol
  dataset           : XAB
  device            : CPU
  gf_or_f           : f
  isotopologue      : 24Mg-1H
  max_memory        : 512
  max_wnl           : 100
  min_wnl           : 0
  molecule          : MgH
  ncpufiles         : 1
  ncputrans         : 4
  output            : memory
  read_path         : /Users/beryl/Academic/UCL/PhD/Data/database/ExoMol/
  refresh_cache     : False
  unc_filter        : 0.5
  wn_wl             : WN
  wn_wl_unit        : cm-1


## 9. Stick spectra

Multiple temperatures are stored by `Condition`. Use `select(T=...)` instead of relying on dictionary order.

In [11]:
stick_result = px.stick_spectra(
    data=data,
    temperatures=[296, 3000],
    abs_emi='Ab',
    threshold=None,
    **FAST_RANGE,
    plot=False,
    **MEMORY,
)
print('conditions:', stick_result.conditions)
stick_296 = stick_result.select(T=296)
stick_3000 = stick_result.select(T=3000)
print(stick_296.head())
print('T=3000 rows:', len(stick_3000))

conditions: [Condition(T=296, P=None, Tvib=None, Trot=None), Condition(T=3000, P=None, Tvib=None, Trot=None)]
          v             S   J'            E'   J"            E"
0  1.706974  2.549769e-45  0.5  10353.980789  0.5  10352.273815
1  1.735588  5.073253e-45  1.5  10354.009403  0.5  10352.273815
4  3.359384  4.179287e-45  1.5  10357.368787  1.5  10354.009403
5  3.387998  2.092688e-44  1.5  10357.368787  0.5  10353.980789
6  3.407896  3.769428e-44  2.5  10357.417299  1.5  10354.009403
T=3000 rows: 353


## 10. Cross sections

The broadening parameters follow `test_api_exomol.py`. `result.params` shows only parameters relevant to cross-section calculation and the active LTE/CPU modes.

In [12]:
XSEC_PARAMS = dict(
    temperatures=[296, 3000],
    pressures=[1.0],
    abs_emi='Ab',
    threshold=1e-30,
    bin_size=0.1,
    profile='Gaussian',
    predissociation=False,
    cutoff=25.0,
    broadeners=['H2', 'He'],
    ratios=[0.85, 0.15],
    gamma_broadening=[0.06, 0.03],
    n_broadening=[0.60, 0.30],
    plot=False,
)

xsec_result = px.cross_sections(
    data=data, **FAST_RANGE, **XSEC_PARAMS, **MEMORY
)
print('conditions:', xsec_result.conditions)
xsec_296 = xsec_result.select(T=296, P=1.0)
print('shape:', xsec_296.shape)
print('units:', xsec_result.units)
print(xsec_result.params)

conditions: [Condition(T=296, P=1.0, Tvib=None, Trot=None), Condition(T=3000, P=1.0, Tvib=None, Trot=None)]
shape: (1001,)
units: {'wavenumber': 'cm-1', 'data': 'cm2/molecule'}
Calculation parameters
  N_point           : 1001
  P_list            : [1.0]
  T_list            : [296, 3000]
  abs_emi           : Ab
  abundance         : 1.0
  bin_size          : 0.1
  broadeners        : ['H2', 'He']
  cache             : parquet
  chunk_size        : 500000
  cutoff            : 25.0
  database          : ExoMol
  dataset           : XAB
  device            : CPU
  doppler_hwhm_yn   : N
  gamma_broadening  : [0.06, 0.03]
  isotopologue      : 24Mg-1H
  lorentzian_hwhm_yn: N
  max_memory        : 512
  max_wnl           : 100
  min_wnl           : 0
  molecule          : MgH
  n_broadening      : [0.6, 0.3]
  ncpufiles         : 1
  ncputrans         : 4
  nlte_method       : L
  output            : memory
  predissoc_yn      : N
  profile           : GAUSSIAN
  ratios            : [0.85,

Select several matches with `squeeze=False`:

In [13]:
matches = xsec_result.select(T=296, squeeze=False)
print(matches.keys())

dict_keys([Condition(T=296, P=1.0, Tvib=None, Trot=None)])


## 11. Combined stick spectra and cross sections

A combined result requires `product=`. Stick spectra do not use pressure; cross sections do. Units are stored separately by product.

In [14]:
combined_result = px.stick_spectra_cross_section(
    data=data, **FAST_RANGE, **XSEC_PARAMS, **MEMORY
)
print('products:', combined_result.data.keys())
print('conditions:', combined_result.conditions)
print('units:', combined_result.units)

combined_stick = combined_result.select(T=296, product='stick_spectra')
combined_xsec = combined_result.select(T=296, P=1.0, product='cross_section')
print('stick rows:', len(combined_stick))
print('cross-section shape:', combined_xsec.shape)

products: dict_keys(['stick_spectra', 'cross_section'])
conditions: {'stick_spectra': [Condition(T=296, P=None, Tvib=None, Trot=None), Condition(T=3000, P=None, Tvib=None, Trot=None)], 'cross_section': [Condition(T=296, P=1.0, Tvib=None, Trot=None), Condition(T=3000, P=1.0, Tvib=None, Trot=None)]}
units: {'stick_spectra': {'wavenumber': 'cm-1', 'data': 'cm/molecule'}, 'cross_section': {'wavenumber': 'cm-1', 'data': 'cm2/molecule'}}
stick rows: 116
cross-section shape: (1001,)


## 12. NLTE selection example

In [15]:
NLTE_PARAMS = dict(
    nlte_method='T',
    tvib_list=[3000],
    trot_list=[296],
    vib_label=['v', 'ElecState'],
    rot_label=['J', 'e/f'],
)

# Uncomment to calculate and select a two-temperature NLTE stick spectrum.
# nlte_result = px.stick_spectra(
#     data=data, **FAST_RANGE, **NLTE_PARAMS,
#     abs_emi='Ab', threshold=1e-30, plot=False, **MEMORY,
# )
# nlte_stick = nlte_result.select(Tvib=3000, Trot=296)

## 13. Output, parameters, and timing summary

Use `output='both'` when the same calculation should return a Result and save files. `calculate` timing excludes result-file saving and plotting.

In [16]:
print(xsec_result.params)
print('profile:', xsec_result.params['profile'])
parameter_dict = xsec_result.params.to_dict()

for phase, elapsed in xsec_result.timing.items():
    print(f"{phase:>10}: CPU={elapsed['cpu']:.6f} s, system={elapsed['system']:.6f} s")

Calculation parameters
  N_point           : 1001
  P_list            : [1.0]
  T_list            : [296, 3000]
  abs_emi           : Ab
  abundance         : 1.0
  bin_size          : 0.1
  broadeners        : ['H2', 'He']
  cache             : parquet
  chunk_size        : 500000
  cutoff            : 25.0
  database          : ExoMol
  dataset           : XAB
  device            : CPU
  doppler_hwhm_yn   : N
  gamma_broadening  : [0.06, 0.03]
  isotopologue      : 24Mg-1H
  lorentzian_hwhm_yn: N
  max_memory        : 512
  max_wnl           : 100
  min_wnl           : 0
  molecule          : MgH
  n_broadening      : [0.6, 0.3]
  ncpufiles         : 1
  ncputrans         : 4
  nlte_method       : L
  output            : memory
  predissoc_yn      : N
  profile           : GAUSSIAN
  ratios            : [0.85, 0.15]
  read_path         : /Users/beryl/Academic/UCL/PhD/Data/database/ExoMol/
  refresh_cache     : False
  threshold         : 1e-30
  unc_filter        : 0.5
  wn_wl       

## 14. Regression checks

In [17]:
assert partition_result.select() is not None
assert 'profile' not in partition_result.params
assert xsec_result.params['profile'] == 'GAUSSIAN'
assert xsec_result.units['data'] == 'cm2/molecule'
assert combined_result.units['stick_spectra']['data'] == 'cm/molecule'
assert combined_result.units['cross_section']['data'] == 'cm2/molecule'
assert {'load', 'prepare', 'calculate', 'save', 'total'} <= set(xsec_result.timing)
print('All Result API examples passed.')

All Result API examples passed.
